# End-to-End Multimodal Music Context Inference Demo
This notebook demonstrates the end-to-end forward pass combining an audio segment graph and text context using the cross-attention fusion architecture.

In [ ]:
import torch
import glob
from transformers import AutoTokenizer, AutoModel
from src.gnn_model import GNNBackbone
from src.fusion_model import CrossAttentionFusionModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
bert = AutoModel.from_pretrained('distilbert-base-uncased')
gnn = GNNBackbone(input_channels=12, hidden_channels=64)
model = CrossAttentionFusionModel(gnn, bert, num_classes=50).to(device)
model.eval()

sample_graph_files = sorted(glob.glob('data/processed/graphs/*.pt'))
graph_data = torch.load(sample_graph_files[0], weights_only=False).to(device)
graph_data.batch = torch.zeros(graph_data.x.size(0), dtype=torch.long, device=device)

text_input = 'Music featuring classical, strings, slow'
enc = tokenizer([text_input], padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)

with torch.no_grad():
    logits, z, attn = model(graph_data, enc['input_ids'], enc['attention_mask'], return_attn=True)
    probs = torch.sigmoid(logits)

print('Inference Successful!')
print(f'Input Text: {text_input}')
print(f'Graph Nodes: {graph_data.x.shape[0]} | Latent z shape: {z.shape}')
print(f'Top predicted class indices: {torch.topk(probs, 3).indices.cpu().numpy().flatten()}')